# Importing

In [1]:
# ── Cell 1 — Config ───────────────────────────────────────────────────────────
import io, re, json, urllib.request
import pandas as pd

panels = {
    "CGN":   ["C3","CD46","CFH","CFHR5","CFI","COL4A3","COL4A4","COL4A5","COL4A6","FN1"],
    "CAKUT": ["ACE","AGT","AGTR1","BMP4","CHD1L","CHRM3","DSTYK","EYA1","FGF20","FRAS1",
              "FREM1","FREM2","GATA3","GRIP1","HNF1B","HPSE2","ITGA8","KAL1","LRIG2","MUC1",
              "PAX2","REN","RET","ROBO2","SALL1","SIX1","SIX2","SIX5","SOX17","SRGAP1",
              "TBX18","TNXB","TRAP1","UMOD","UPK3A","WNT4"],
    "SRNS":  ["ACTN4","ADCK4","ANLN","ARHGAP24","ARHGDIA","CD2AP","COQ2","COQ6","CRB2",
              "CUBN","DGKE","EMP2","FAT1","INF2","ITGA3","ITGB4","KANK1","KANK2","KANK4",
              "LAMB2","LMX1B","MTTL1","MYH9","MYO1E","NPHS1","NPHS2","NUP107","NUP205",
              "NUP93","PDSS2","PLCE1","PTPRO","SCARB2","SMARCAL1","TRPC6","WDR73","WT1","XPO5"],
    "USD":   ["ADCY10","AGXT","APRT","ATP6V0A4","ATP6V1B1","CA2","CASR","CLCN5","CLCNKB",
              "CLDN16","CLDN19","CYP24A1","FAM20A","GRHPR","HNF4A","HOGA1","HPRT1","KCNJ1",
              "OCRL","SLC12A1","SLC22A12","SLC2A9","SLC34A1","SLC34A3","SLC3A1","SLC4A1",
              "SLC7A9","SLC9A3R1","VDR","XDH"],
    "NPHP":  ["ANKS6","CEP164","CEP290","GLIS2","INVS","IQCB1","NEK8","NPHP1","NPHP3",
              "NPHP4","RPGRIP1L","SDCCAG8","TMEM67","TTC21B","WDR19","ZNF423"],
}

# Verified UniProt accessions — 129 genes, MTTL1 excluded (mitochondrial tRNA)
GENE_TO_UNIPROT = {
    "C3":"P01024","CD46":"P15529","CFH":"P08603","CFHR5":"Q9BXR6","CFI":"P05156",
    "COL4A3":"Q01955","COL4A4":"P53420","COL4A5":"P29400","COL4A6":"P25067","FN1":"P02751",
    "ACE":"P12821","AGT":"P01019","AGTR1":"P30556","BMP4":"P12644","CHD1L":"Q86WJ1",
    "CHRM3":"P20309","DSTYK":"Q6XUX3","EYA1":"Q99502","FGF20":"Q9NP95","FRAS1":"Q86WK7",
    "FREM1":"A4FU01","FREM2":"O94898","GATA3":"P23771","GRIP1":"O14490","HNF1B":"P35680",
    "HPSE2":"Q8WWQ2","ITGA8":"P53708","KAL1":"P23352","LRIG2":"Q6UXZ3","MUC1":"P15941",
    "PAX2":"Q02962","REN":"P00797","RET":"P07949","ROBO2":"O94813","SALL1":"Q9NSC2",
    "SIX1":"Q15475","SIX2":"Q9NPC8","SIX5":"Q9UBR2","SOX17":"Q9Y458","SRGAP1":"O94988",
    "TBX18":"O95935","TNXB":"P22105","TRAP1":"Q12931","UMOD":"P07911","UPK3A":"O75631",
    "WNT4":"P56705","ACTN4":"O43707","ADCK4":"Q96D53","ANLN":"Q9NQW6","ARHGAP24":"Q8N264",
    "ARHGDIA":"P52565","CD2AP":"Q9Y5K6","COQ2":"Q96H96","COQ6":"Q9Y2Z9","CRB2":"Q5IJ48",
    "CUBN":"O60494","DGKE":"P52429","EMP2":"P55001","FAT1":"Q14517","INF2":"Q27J81",
    "ITGA3":"P26006","ITGB4":"P16144","KANK1":"Q14678","KANK2":"Q2M1Z3","KANK4":"Q5T7N3",
    "LAMB2":"P55268","LMX1B":"O15146","MYH9":"P35579","MYO1E":"Q12965","NPHS1":"O60500",
    "NPHS2":"Q9NP85","NUP107":"P57740","NUP205":"Q92621","NUP93":"Q8N1F7","PDSS2":"Q86YH6",
    "PLCE1":"Q9P212","PTPRO":"P23471","SCARB2":"Q14108","SMARCAL1":"Q9NZC9","TRPC6":"Q9Y210",
    "WDR73":"Q8WV93","WT1":"P19544","XPO5":"Q9HAV4","ADCY10":"Q96PN6","AGXT":"P21549",
    "APRT":"P07741","ATP6V0A4":"Q9HBG4","ATP6V1B1":"P15313","CA2":"P00918","CASR":"P41180",
    "CLCN5":"P51795","CLCNKB":"P51800","CLDN16":"Q9Y5I7","CLDN19":"Q8N6F1","CYP24A1":"Q07973",
    "FAM20A":"Q5T9L3","GRHPR":"Q9UBQ7","HNF4A":"P41235","HOGA1":"Q86YQ8","HPRT1":"P00492",
    "KCNJ1":"P48549","OCRL":"Q01968","SLC12A1":"Q13621","SLC22A12":"Q9UHW9","SLC2A9":"Q9NRM0",
    "SLC34A1":"Q06495","SLC34A3":"Q8N130","SLC3A1":"Q07837","SLC4A1":"P02730","SLC7A9":"P82251",
    "SLC9A3R1":"O14745","VDR":"P11473","XDH":"P47989","ANKS6":"O15178","CEP164":"Q9UKJ0",
    "CEP290":"O15078","GLIS2":"Q9BZE0","INVS":"Q9Y283","IQCB1":"P58876","NEK8":"Q86SG6",
    "NPHP1":"O15259","NPHP3":"Q7Z494","NPHP4":"O75161","RPGRIP1L":"Q68CZ1","SDCCAG8":"Q86SQ7",
    "TMEM67":"Q5HYA8","TTC21B":"Q7Z4B0","WDR19":"Q8IV38","ZNF423":"Q2M1K9",
}

gene_panel       = {g: p for p, gs in panels.items() for g in gs}
all_genes        = [g for gs in panels.values() for g in gs]
all_genes_no_mttl = [g for g in all_genes if g != "MTTL1"]
UNIPROT_TO_GENE  = {v: k for k, v in GENE_TO_UNIPROT.items()}

# ── Sanity checks ─────────────────────────────────────────────────────────────
from collections import Counter
assert sum(len(gs) for gs in panels.values()) == 130, "Expected 130 total genes"
assert len(all_genes_no_mttl) == 129, "Expected 129 genes excl. MTTL1"
assert len(GENE_TO_UNIPROT) == 129, "Expected 129 UniProt entries"
dups = [u for u, c in Counter(GENE_TO_UNIPROT.values()).items() if c > 1]
assert len(dups) == 0, f"Duplicate UniProts: {dups}"

print(f"Total genes          : {sum(len(gs) for gs in panels.values())}")
print(f"Excl. MTTL1          : {len(all_genes_no_mttl)}")
print(f"Unique UniProt IDs   : {len(set(GENE_TO_UNIPROT.values()))}")
print(f"Panel counts         : { {p: len(gs) for p, gs in panels.items()} }")
print("All checks passed ✓")

clinvar_base = "https://raw.githubusercontent.com/Joshua-Pillai/NephVar/main"
rsa_base     = "https://raw.githubusercontent.com/NephVar/NephVar/main/biophysical"

Total genes          : 130
Excl. MTTL1          : 129
Unique UniProt IDs   : 129
Panel counts         : {'CGN': 10, 'CAKUT': 36, 'SRNS': 38, 'USD': 30, 'NPHP': 16}
All checks passed ✓


In [2]:
# ── Cell 2 — Load RSA (tolerate missing genes) ────────────────────────────────
rows, errors = [], []
for gene in all_genes_no_mttl:
    url = f"{rsa_base}/{gene}.html"
    try:
        with urllib.request.urlopen(url, timeout=15) as r:
            html = r.read().decode("utf-8", errors="replace")
        m = re.search(r'const residues = (\[.*?\]);', html, re.DOTALL)
        if not m: raise ValueError("residues block not found")
        for res in json.loads(m.group(1)):
            rows.append({"gene": gene, "resnum": res["resnum"],
                         "rsa": res["rsa"], "ss": res["ss"], "plddt": res["plddt"]})
    except Exception as e:
        errors.append(gene)  # just log, don't crash

df_rsa       = pd.DataFrame(rows)
rsa_residues = set(zip(df_rsa["gene"], df_rsa["resnum"]))
print(f"Genes loaded  : {df_rsa['gene'].nunique()} / {len(all_genes_no_mttl)}")
print(f"Total residues: {len(df_rsa):,}")
print(f"Errors (no RSA page): {errors}")



Genes loaded  : 129 / 129
Total residues: 127,472
Errors (no RSA page): []


In [4]:
# ── Cell 3 — Functions + Load ClinVar + ACMG bucket + join RSA ───────────────
import io, re
import pandas as pd

PATHOGENIC = {"Pathogenic","Likely pathogenic","Pathogenic/Likely pathogenic",
              "Pathogenic, low penetrance","Likely pathogenic, low penetrance",
              "Pathogenic/Likely pathogenic, low penetrance",
              "Likely pathogenic/Likely pathogenic, low penetrance",
              "Likely pathogenic/Pathogenic, low penetrance"}
VUS_SET    = {"Uncertain significance","Uncertain significance/Uncertain risk allele",
              "Uncertain risk allele","Likely pathogenic/Likely risk allele",
              "Likely risk allele","Uncertain significance/VUS-mid"}
BENIGN     = {"Benign","Likely benign","Benign/Likely benign"}

def bucket(label):
    if pd.isna(label): return "Other"
    label = str(label).strip()
    if label in PATHOGENIC: return "P" if label == "Pathogenic" else "LP"
    if label in VUS_SET:    return "VUS"
    if label in BENIGN:     return "B" if label == "Benign" else "LB"
    base = label.split(";")[0].strip()
    if base == "Pathogenic":        return "P"
    if base == "Likely pathogenic": return "LP"
    if base == "Benign":            return "B"
    if base == "Likely benign":     return "LB"
    if "VUS" in label or "Uncertain significance" in label: return "VUS"
    return "Other"

def parse_resnum(protein_change, gene):
    if pd.isna(protein_change): return None
    first_parsed = None
    for part in str(protein_change).split(','):
        m = re.search(r'[A-Za-z*](\d+)', part.strip())
        if m:
            resnum = int(m.group(1))
            if first_parsed is None: first_parsed = resnum
            if (gene, resnum) in rsa_residues: return resnum
    return first_parsed

# ── Load ClinVar ──────────────────────────────────────────────────────────────
rows = []
for panel, genes in panels.items():
    for gene in genes:
        url = f"{clinvar_base}/{panel}/{gene}.txt"
        try:
            with urllib.request.urlopen(url, timeout=15) as r:
                content = r.read().decode("utf-8", errors="replace")
            df_g = pd.read_csv(io.StringIO(content), sep="\t", low_memory=False)
            df_g["gene"]  = gene
            df_g["panel"] = panel
            rows.append(df_g)
        except Exception as e:
            print(f"  ERROR {gene}: {e}")

df_raw = pd.concat(rows, ignore_index=True)
n_dup  = df_raw["VariationID"].duplicated(keep="first").sum()
df_raw = df_raw.drop_duplicates(subset="VariationID", keep="first").copy()
print(f"Raw: {len(df_raw)+n_dup:,}  |  Dupes: {n_dup:,}  |  Unique: {len(df_raw):,}")

# ── Missense, parse positions, bucket, join RSA ───────────────────────────────
df_miss = df_raw[
    df_raw["Molecular consequence"].str.contains("missense", case=False, na=False)
].copy()
df_miss["resnum"] = df_miss.apply(
    lambda r: parse_resnum(r["Protein change"], r["gene"]), axis=1
)
df_miss["acmg"] = df_miss["Germline classification"].apply(bucket)

df_merged    = df_miss[df_miss["resnum"].notna() & (df_miss["acmg"] != "Other")].copy()
df_merged    = df_merged.merge(df_rsa[["gene","resnum","rsa","ss","plddt"]],
                               on=["gene","resnum"], how="left")
df_rsa_clean = df_merged[df_merged["rsa"].notna()].copy()

print(f"\nACMG counts (RSA-matched missense):")
print(df_rsa_clean["acmg"].value_counts().reindex(["P","LP","VUS","LB","B"]).to_string())

df_vus = df_rsa_clean[df_rsa_clean["acmg"] == "VUS"].copy()
print(f"\nMissense VUS (RSA-matched): {len(df_vus):,}")

Raw: 119,315  |  Dupes: 1,942  |  Unique: 117,373

ACMG counts (RSA-matched missense):
acmg
P        895
LP      2609
VUS    44021
LB      2230
B        591

Missense VUS (RSA-matched): 44,021


In [5]:
# ── Cell 4 — Stream AlphaMissense for all 129 genes ───────────────────────────
import gzip

AM_URL      = "https://zenodo.org/records/8208688/files/AlphaMissense_aa_substitutions.tsv.gz"
our_uniprots = set(GENE_TO_UNIPROT.values())

print(f"Unique UniProt IDs : {len(our_uniprots)}  (expect 129)")
print("Streaming AlphaMissense (~3 min)...")

am_rows, header, n_comment = [], None, 0

with urllib.request.urlopen(AM_URL, timeout=300) as r:
    with gzip.open(r, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith("#"):
                n_comment += 1
                continue
            parts = line.split("\t")
            if header is None:
                header = parts
                print(f"Columns: {header}")
                continue
            if len(parts) != len(header):
                continue
            if parts[0] in our_uniprots:
                am_rows.append(parts)

df_am = pd.DataFrame(am_rows, columns=header)
df_am.rename(columns={
    header[0]: "uniprot",
    header[1]: "protein_variant",
    header[2]: "am_pathogenicity",
    header[3]: "am_class"
}, inplace=True)
df_am["am_pathogenicity"] = pd.to_numeric(df_am["am_pathogenicity"], errors="coerce")
df_am["gene"]   = df_am["uniprot"].map(UNIPROT_TO_GENE)
df_am["resnum"] = df_am["protein_variant"].str.extract(r'[A-Z](\d+)[A-Z]').astype(float)

print(f"\nGenes covered : {df_am['gene'].nunique()} / 129")
print(f"Total variants: {len(df_am):,}")
print(f"\nAM class distribution:")
print(df_am["am_class"].value_counts().to_string())

df_am.to_csv("NephVar_AlphaMissense.tsv.gz", sep="\t", index=False, compression="gzip")
print("\nSaved: NephVar_AlphaMissense.tsv.gz → upload to NephVar/Data/")

Unique UniProt IDs : 129  (expect 129)
Streaming AlphaMissense (~3 min)...
Columns: ['uniprot_id', 'protein_variant', 'am_pathogenicity', 'am_class']

Genes covered : 129 / 129
Total variants: 2,230,448

AM class distribution:
am_class
pathogenic    978951
benign        939853
ambiguous     311644

Saved: NephVar_AlphaMissense.tsv.gz → upload to NephVar/Data/


In [18]:
import zipfile, os

os.makedirs("am_by_gene", exist_ok=True)

for gene in all_genes_no_mttl:
    gene_df = df_am[df_am["gene"] == gene]
    if len(gene_df) > 0:
        gene_df.to_csv(f"am_by_gene/{gene}_AM.csv", index=False)

# Zip them all
with zipfile.ZipFile("NephVar_AM_by_gene.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for gene in all_genes_no_mttl:
        fpath = f"am_by_gene/{gene}_AM.csv"
        if os.path.exists(fpath):
            zf.write(fpath, f"{gene}_AM.csv")

size_mb = os.path.getsize("NephVar_AM_by_gene.zip") / 1e6
print(f"Saved: NephVar_AM_by_gene.zip ({size_mb:.1f} MB)")
print(f"Files: {len([g for g in all_genes_no_mttl if os.path.exists(f'am_by_gene/{g}_AM.csv')])}/129")

Saved: NephVar_AM_by_gene.zip (14.1 MB)
Files: 129/129
